# 03 - Daily Activities Data Wrangling

Notebook ini memproses dataset `daily_activities.csv`.

Tabel ini menjadi pusat proses wrangling karena merepresentasikan input aktivitas harian pengguna. Data yang berasal dari input user lebih mungkin memiliki nilai kosong, format tidak konsisten, duplikasi pengisian, dan nilai yang tidak sesuai batas logis.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Mengatur tampilan dataframe agar output notebook lebih mudah dibaca.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Cari root project otomatis
# Mengambil lokasi kerja notebook saat ini.
current_path = Path.cwd().resolve()

# Menelusuri parent folder sampai menemukan root project yang memiliki folder data/raw.
for path in [current_path] + list(current_path.parents):
    if (path / "data" / "raw").exists():
        PROJECT_ROOT = path
        break

# Menentukan folder sumber data raw.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
# Menentukan folder output data hasil cleaning.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# Menentukan folder output report dan validation summary.
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

# Membuat folder processed jika belum tersedia.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# Membuat folder reports jika belum tersedia.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Menampilkan path project untuk memastikan notebook membaca folder yang benar.
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORT_DIR   :", REPORT_DIR)

PROJECT_ROOT : C:\Data Codingan\student_stress_data_science
RAW_DIR      : C:\Data Codingan\student_stress_data_science\data\raw
PROCESSED_DIR: C:\Data Codingan\student_stress_data_science\data\processed
REPORT_DIR   : C:\Data Codingan\student_stress_data_science\outputs\reports


## 1. Load Dataset

In [2]:
# memuat dataset dari folder yang sesuai dan menampilkan sampel awal data.
# Membaca file CSV ke dalam dataframe.
daily_activities = pd.read_csv(RAW_DIR / "daily_activities.csv")
users_clean = pd.read_csv(PROCESSED_DIR / "users_clean.csv")

# Menampilkan beberapa baris awal untuk memahami bentuk data.
daily_activities.head()

,id,user_id,activity_date,sleep_hours,study_hours,screen_time_hours,social_media_hours,physical_activity_minutes,caffeine_intake_mg,mood_score,fatigue_level,assignment_load,deadline_pressure,social_interaction_score,financial_worry_score,health_condition_score,created_at,updated_at
0,1,1,2026-01-01,7.23,1.93,5.78,3.81,13.0,160,4,7,5,4,6,2,5,2026-01-01 22:45:00,2026-01-01 23:39:00
1,2,1,2026-01-02,NaN,2.70,7.88,3.78,21.0,105,6,5,9,6,4,5,4,2026-01-02 19:44:00,2026-01-02 20:40:00
2,3,1,2026-01-03,6.56,3.24,8.65,4.78,23.0,123,6,6,5,5,8,2,6,2026-01-03 20:28:00,2026-01-03 20:33:00
3,4,1,2026-01-04,7.17,2.37,8.52,5.34,36.0,91,3,7,6,7,7,3,5,2026-01-04 23:55:00,2026-01-05 00:40:00
4,5,1,2026-01-05,7.6,5.20,7.43,4.82,29.0,103,6,6,7,9,2,5,6,2026-01-05 22:27:00,2026-01-05 23:02:00


## 2. Assessing Data

Pemeriksaan difokuskan pada missing value, duplicate berdasarkan `user_id + activity_date`, format tanggal, format numerik, dan potensi nilai yang tidak sesuai batas wajar.

Karena tabel ini akan menjadi sumber fitur utama, kesalahan pada tahap ini dapat memengaruhi kualitas analisis dan modelling.

In [3]:
# Menampilkan struktur kolom, tipe data, dan jumlah non-null.
daily_activities.info()

<class 'pandas.DataFrame'>
RangeIndex: 27405 entries, 0 to 27404
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         27405 non-null  int64  
 1   user_id                    27405 non-null  int64  
 2   activity_date              27405 non-null  str    
 3   sleep_hours                26857 non-null  str    
 4   study_hours                27405 non-null  float64
 5   screen_time_hours          27405 non-null  str    
 6   social_media_hours         26329 non-null  float64
 7   physical_activity_minutes  26591 non-null  float64
 8   caffeine_intake_mg         26037 non-null  str    
 9   mood_score                 27405 non-null  int64  
 10  fatigue_level              27405 non-null  int64  
 11  assignment_load            27405 non-null  int64  
 12  deadline_pressure          27405 non-null  int64  
 13  social_interaction_score   27405 non-null  int64  
 14  f

In [4]:
# Menampilkan ringkasan statistik untuk kolom numerik dan kategorikal.
daily_activities.describe(include='all')

,id,user_id,activity_date,sleep_hours,study_hours,screen_time_hours,social_media_hours,physical_activity_minutes,caffeine_intake_mg,mood_score,fatigue_level,assignment_load,deadline_pressure,social_interaction_score,financial_worry_score,health_condition_score,created_at,updated_at
count,27405.0000,27405.000000,27405,26857,27405.000000,27405,26329.000000,26591.000000,26037,27405.000000,27405.000000,27405.000000,27405.000000,27405.000000,27405.000000,27405.000000,27405,27405
unique,NaN,NaN,339,988,NaN,1435,NaN,NaN,795,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18418,19179
top,NaN,NaN,2026-03-06,28,NaN,7.18,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-28 23:01:00,2026-01-01 23:55:00
freq,NaN,NaN,304,135,NaN,82,NaN,NaN,1033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,8
mean,13703.0000,150.518883,NaN,NaN,4.110956,NaN,3.063609,27.723440,NaN,4.392848,6.100383,6.640029,6.392885,6.397519,5.182339,6.093341,NaN,NaN
std,7911.2864,86.539895,NaN,NaN,1.656462,NaN,2.119260,17.565747,NaN,1.469842,1.518639,1.774117,1.900119,1.729177,1.875600,1.216902,NaN,NaN
min,1.0000,1.000000,NaN,NaN,0.000000,NaN,0.000000,-5.000000,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN
25%,6852.0000,76.000000,NaN,NaN,2.960000,NaN,2.160000,14.000000,NaN,3.000000,5.000000,5.000000,5.000000,5.000000,4.000000,5.000000,NaN,NaN
50%,13703.0000,150.000000,NaN,NaN,4.060000,NaN,2.920000,27.000000,NaN,4.000000,6.000000,7.000000,6.000000,6.000000,5.000000,6.000000,NaN,NaN
75%,20554.0000,225.000000,NaN,NaN,5.200000,NaN,3.680000,40.000000,NaN,5.000000,7.000000,8.000000,8.000000,8.000000,6.000000,7.000000,NaN,NaN


In [5]:
# menilai missing value, duplicate, dan kandidat masalah kualitas data.
print("Missing value:")
# Menghitung jumlah missing value pada setiap kolom.
print(daily_activities.isna().sum())

# Mengecek keberadaan data duplicate berdasarkan aturan yang relevan.
print("\nDuplicate full rows:", daily_activities.duplicated().sum())
print("Duplicate user_id + activity_date:", daily_activities.duplicated(["user_id", "activity_date"]).sum())

Missing value:
id                              0
user_id                         0
activity_date                   0
sleep_hours                   548
study_hours                     0
screen_time_hours               0
social_media_hours           1076
physical_activity_minutes     814
caffeine_intake_mg           1368
mood_score                      0
fatigue_level                   0
assignment_load                 0
deadline_pressure               0
social_interaction_score        0
financial_worry_score           0
health_condition_score          0
created_at                      0
updated_at                      0
dtype: int64

Duplicate full rows: 0
Duplicate user_id + activity_date: 405


In [6]:
# menjalankan bagian kode pada tahap ini sesuai konteks notebook.
columns_to_check = [
    "activity_date",
    "sleep_hours",
    "screen_time_hours",
    "caffeine_intake_mg"
]

for col in columns_to_check:
    print(f"\nSample unique values from {col}:")
    # Melihat variasi nilai unik untuk menilai konsistensi kategori atau format.
    print(daily_activities[col].dropna().astype(str).unique()[:15])


Sample unique values from activity_date:
<StringArray>
['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04', '2026-01-05',
 '2026-01-06', '2026-01-07', '2026-01-08', '2026-01-09', '2026-01-10',
 '2026-01-11', '2026-01-12', '2026-01-13', '2026-01-14', '2026-01-15']
Length: 15, dtype: str

Sample unique values from sleep_hours:
<StringArray>
['7.23', '6.56', '7.17',  '7.6', '6.88', '7.14', '6.55', '7.05',  '7.1',
 '6.38', '6.61', '6.91', '6.52', '7.19', '7.41']
Length: 15, dtype: str

Sample unique values from screen_time_hours:
<StringArray>
[ '5.78',  '7.88',  '8.65',  '8.52',  '7.43',  '5.52',   '6.8',  '7.91',
 '7.98h',  '8.17',  '7.98',  '6.62',  '7.99', '11.36', '8.59h']
Length: 15, dtype: str

Sample unique values from caffeine_intake_mg:
<StringArray>
['160', '105', '123',  '91', '103', '230', '157',  '87',  '86',  '62', '170',
  '73', '199', '207', '196']
Length: 15, dtype: str


## Insight:

Hasil assessing menunjukkan bahwa `daily_activities` memerlukan cleaning paling intensif dibandingkan tabel lain. Tabel ini memiliki risiko format campuran, seperti angka yang ditulis bersama satuan, tanggal yang tidak selalu konsisten, serta duplicate submit untuk user dan tanggal yang sama.

Secara analitik, duplicate pada kombinasi `user_id + activity_date` tidak boleh dipertahankan karena satu user hanya seharusnya memiliki satu catatan aktivitas untuk satu hari tertentu. Jika terdapat lebih dari satu record, record terbaru berdasarkan `updated_at` dipilih karena dianggap merepresentasikan revisi terakhir dari input user.

Tindakan cleaning yang dilakukan meliputi parsing tanggal, parsing numerik, imputasi missing value dengan median per user dan median global, pembatasan range nilai, perbaikan inkonsistensi logis, serta deduplikasi.

## 3. Helper Function

In [7]:
# mendefinisikan fungsi bantu lokal yang digunakan pada notebook ini.
def parse_number(value):
    # Mengubah nilai seperti '7 jam', '8h', atau '150 mg' menjadi angka.
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text == "":
        return np.nan

    match = re.search(r"-?\d+(\.\d+)?", text)

    if match:
        return float(match.group(0))

    return np.nan


def parse_date(value):
    # Mengubah beberapa format tanggal menjadi datetime.
    if pd.isna(value):
        return pd.NaT

    text = str(value).strip()

    if text == "":
        return pd.NaT

    # Mengubah kolom ke tipe datetime; format yang tidak valid menjadi NaT.
    date = pd.to_datetime(text, errors="coerce")

    if pd.isna(date):
        # Mengubah kolom ke tipe datetime; format yang tidak valid menjadi NaT.
        date = pd.to_datetime(text, errors="coerce", dayfirst=True)

    return date

## 4. Cleaning Data

Strategi cleaning:

1. Key dan tanggal diubah ke tipe data yang sesuai.
2. Kolom numerik diparsing dari format campuran menjadi angka.
3. Baris dengan key atau tanggal fatal yang tidak valid dihapus.
4. Hanya `user_id` yang valid terhadap `users_clean` yang dipertahankan.
5. Missing value pada fitur numerik diisi menggunakan median per user; jika masih kosong, digunakan median global.
6. Kolom jam dibatasi pada rentang 0 sampai 24.
7. Kolom skor dibatasi pada rentang 1 sampai 10.
8. Nilai negatif pada aktivitas fisik dan kafein diperbaiki ke batas bawah 0.
9. `social_media_hours` disesuaikan agar tidak melebihi `screen_time_hours`.
10. Duplicate `user_id + activity_date` diselesaikan dengan mengambil record terbaru.

In [8]:
# menilai missing value, duplicate, dan kandidat masalah kualitas data.
daily_clean = daily_activities.copy()
raw_daily_rows = len(daily_clean)

# Mengubah kolom ke tipe numerik; nilai yang gagal dikonversi menjadi NaN.
daily_clean["id"] = pd.to_numeric(daily_clean["id"], errors="coerce")
daily_clean["user_id"] = pd.to_numeric(daily_clean["user_id"], errors="coerce")
daily_clean["activity_date"] = daily_clean["activity_date"].apply(parse_date)

hour_columns = [
    "sleep_hours",
    "study_hours",
    "screen_time_hours",
    "social_media_hours"
]

score_columns = [
    "mood_score",
    "fatigue_level",
    "assignment_load",
    "deadline_pressure",
    "social_interaction_score",
    "financial_worry_score",
    "health_condition_score"
]

other_numeric_columns = [
    "physical_activity_minutes",
    "caffeine_intake_mg"
]

numeric_columns = hour_columns + score_columns + other_numeric_columns

for col in numeric_columns:
    daily_clean[col] = daily_clean[col].apply(parse_number)

# Mengubah kolom ke tipe datetime; format yang tidak valid menjadi NaT.
daily_clean["created_at"] = pd.to_datetime(daily_clean["created_at"], errors="coerce")
daily_clean["updated_at"] = pd.to_datetime(daily_clean["updated_at"], errors="coerce")

# Menghapus baris yang kehilangan kolom kunci atau informasi penting.
daily_clean = daily_clean.dropna(subset=["id", "user_id", "activity_date"])

valid_user_ids = set(users_clean["id"])
# Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
daily_clean = daily_clean[daily_clean["user_id"].isin(valid_user_ids)]

# Menghitung jumlah missing value pada setiap kolom.
missing_before_imputation = daily_clean[numeric_columns].isna().sum()
missing_before_imputation

C:\Users\User\AppData\Local\Temp\ipykernel_26500\3702542808.py:31: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  date = pd.to_datetime(text, errors="coerce")


sleep_hours                   548
study_hours                     0
screen_time_hours               0
social_media_hours           1076
mood_score                      0
fatigue_level                   0
assignment_load                 0
deadline_pressure               0
social_interaction_score        0
financial_worry_score           0
health_condition_score          0
physical_activity_minutes     814
caffeine_intake_mg           1368
dtype: int64

In [9]:
# menjalankan bagian kode pada tahap ini sesuai konteks notebook.
for col in numeric_columns:
    median_per_user = daily_clean.groupby("user_id")[col].transform("median")
    global_median = daily_clean[col].median()

    # Mengisi nilai kosong berdasarkan strategi yang ditentukan.
    daily_clean[col] = daily_clean[col].fillna(median_per_user)
    daily_clean[col] = daily_clean[col].fillna(global_median)

for col in hour_columns:
    # Membatasi nilai agar tetap berada pada rentang logis.
    daily_clean[col] = daily_clean[col].clip(0, 24)

for col in score_columns:
    # Membatasi nilai agar tetap berada pada rentang logis.
    daily_clean[col] = daily_clean[col].clip(1, 10)

# Membatasi nilai agar tetap berada pada rentang logis.
daily_clean["physical_activity_minutes"] = daily_clean["physical_activity_minutes"].clip(lower=0)
daily_clean["caffeine_intake_mg"] = daily_clean["caffeine_intake_mg"].clip(lower=0)

daily_clean["social_media_hours"] = np.minimum(
    daily_clean["social_media_hours"],
    daily_clean["screen_time_hours"]
)

# Mengisi nilai kosong berdasarkan strategi yang ditentukan.
daily_clean["sort_time"] = daily_clean["updated_at"].fillna(daily_clean["created_at"])
# Mengurutkan data agar proses deduplikasi atau output lebih stabil.
daily_clean = daily_clean.sort_values(["user_id", "activity_date", "sort_time"])

# Mengecek keberadaan data duplicate berdasarkan aturan yang relevan.
duplicates_removed = daily_clean.duplicated(
    subset=["user_id", "activity_date"],
    keep="last"
).sum()

# Menghapus duplicate sesuai subset key yang ditentukan.
daily_clean = daily_clean.drop_duplicates(
    subset=["user_id", "activity_date"],
    keep="last"
)

# ID tidak di-reassign supaya relasi ke stress_predictions.activity_id tetap bisa divalidasi.
# Mengurutkan data agar proses deduplikasi atau output lebih stabil.
daily_clean = daily_clean.sort_values(["user_id", "activity_date"]).reset_index(drop=True)

daily_clean["id"] = daily_clean["id"].astype(int)
daily_clean["user_id"] = daily_clean["user_id"].astype(int)
daily_clean["activity_date"] = daily_clean["activity_date"].dt.strftime("%Y-%m-%d")

for col in hour_columns:
    # Membulatkan nilai numerik agar format output lebih rapi.
    daily_clean[col] = daily_clean[col].round(2)

for col in score_columns + other_numeric_columns:
    # Membulatkan nilai numerik agar format output lebih rapi.
    daily_clean[col] = daily_clean[col].round().astype(int)

daily_clean["created_at"] = daily_clean["created_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
daily_clean["updated_at"] = daily_clean["updated_at"].dt.strftime("%Y-%m-%d %H:%M:%S")

daily_clean = daily_clean[
    [
        "id", "user_id", "activity_date",
        "sleep_hours", "study_hours", "screen_time_hours", "social_media_hours",
        "physical_activity_minutes", "caffeine_intake_mg",
        "mood_score", "fatigue_level", "assignment_load", "deadline_pressure",
        "social_interaction_score", "financial_worry_score", "health_condition_score",
        "created_at", "updated_at"
    ]
]

print("Raw rows:", raw_daily_rows)
print("Clean rows:", len(daily_clean))
print("Duplicates removed:", duplicates_removed)

# Menampilkan beberapa baris awal untuk memahami bentuk data.
daily_clean.head()

Raw rows: 27405
Clean rows: 26984
Duplicates removed: 421


,id,user_id,activity_date,sleep_hours,study_hours,screen_time_hours,social_media_hours,physical_activity_minutes,caffeine_intake_mg,mood_score,fatigue_level,assignment_load,deadline_pressure,social_interaction_score,financial_worry_score,health_condition_score,created_at,updated_at
0,1,1,2026-01-01,7.23,1.93,5.78,3.81,13,160,4,7,5,4,6,2,5,2026-01-01 22:45:00,2026-01-01 23:39:00
1,2,1,2026-01-02,7.10,2.70,7.88,3.78,21,105,6,5,9,6,4,5,4,2026-01-02 19:44:00,2026-01-02 20:40:00
2,3,1,2026-01-03,6.56,3.24,8.65,4.78,23,123,6,6,5,5,8,2,6,2026-01-03 20:28:00,2026-01-03 20:33:00
3,4,1,2026-01-04,7.17,2.37,8.52,5.34,36,91,3,7,6,7,7,3,5,2026-01-04 23:55:00,2026-01-05 00:40:00
4,5,1,2026-01-05,7.60,5.20,7.43,4.82,29,103,6,6,7,9,2,5,6,2026-01-05 22:27:00,2026-01-05 23:02:00


## Insight Setelah Cleaning:

Dataset `daily_activities_clean` sudah lebih stabil untuk digunakan sebagai sumber fitur harian. Duplicate harian diselesaikan, format angka dan tanggal distandardisasi, serta nilai fitur dipastikan berada pada rentang yang logis.

Baris yang tidak lolos validasi key atau relasi user tidak dipertahankan karena dapat mengganggu konsistensi tabel. Hasil cleaning ini menjadi dasar bagi validasi tabel output yang menggunakan `activity_id`.

## Insight:
`daily_activities_clean` sudah dibersihkan tanpa mengubah ID asli. Ini penting supaya relasi ke `stress_predictions.activity_id` tetap bisa divalidasi.

## 5. Validation dan Save Output

In [10]:
# Tujuan cell: membuat tabel validasi untuk memastikan hasil cleaning memenuhi aturan kualitas data.
# Membuat dataframe validasi untuk mendokumentasikan hasil pengecekan kualitas data.
validation = pd.DataFrame([
    {"rule": "daily_activities.id unique", "passed": daily_clean["id"].is_unique},
    # Mengecek keberadaan data duplicate berdasarkan aturan yang relevan.
    {"rule": "daily_activities user_id + activity_date unique", "passed": not daily_clean.duplicated(["user_id", "activity_date"]).any()},
    {"rule": "social_media_hours <= screen_time_hours", "passed": (daily_clean["social_media_hours"] <= daily_clean["screen_time_hours"]).all()},
    {"rule": "user_id exists in users", "passed": set(daily_clean["user_id"]).issubset(set(users_clean["id"]))},
])

validation

,rule,passed
0,daily_activities.id unique,True
1,daily_activities user_id + activity_date unique,True
2,social_media_hours <= screen_time_hours,True
3,user_id exists in users,True


In [11]:
# Tujuan cell: menyimpan output hasil cleaning atau report ke folder tujuan.
# Menyimpan dataframe ke file CSV.
daily_clean.to_csv(PROCESSED_DIR / "daily_activities_clean.csv", index=False)
validation.to_csv(REPORT_DIR / "daily_activities_validation.csv", index=False)

print("Saved:", PROCESSED_DIR / "daily_activities_clean.csv")

Saved: C:\Data Codingan\student_stress_data_science\data\processed\daily_activities_clean.csv
